# PULSE Temporal Awareness Training
**Fine-tune Qwen 2.5 1.5B with PULSE temporal awareness on free Colab GPU.**

This notebook:
1. Generates 2000 synthetic temporal training examples
2. Fine-tunes Qwen 2.5 1.5B with LoRA (8.8M trainable params)
3. Uploads the model to Hugging Face Hub

Runtime: ~20-30 min on T4 GPU

---

In [ ]:
# Step 0: Install dependencies
!pip install -q torch transformers peft trl datasets accelerate numpy huggingface_hub

In [ ]:
# Step 0b: Login to HF (to upload the trained model)
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Step 1: PULSE inline encoder + data generator
import json, random, numpy as np
from datetime import datetime, timedelta

_COG = np.array([0.20,0.15,0.12,0.10,0.12,0.18,0.30,0.50,0.70,0.85,0.95,0.92,
                 0.85,0.72,0.68,0.75,0.88,0.90,0.82,0.70,0.55,0.40,0.30,0.25])
_NRG = np.array([0.15,0.10,0.08,0.07,0.10,0.20,0.40,0.60,0.75,0.85,0.90,0.88,
                 0.82,0.70,0.65,0.72,0.85,0.88,0.80,0.65,0.50,0.35,0.25,0.18])

def _interp(c, h):
    h0 = int(h) % 24
    return float(c[h0]*(1-(h-int(h))) + c[(h0+1)%24]*(h-int(h)))

def get_phase(h):
    if 6<=h<10: return 'morning_ramp'
    if 10<=h<12: return 'morning_peak'
    if 12<=h<14: return 'post_lunch_dip'
    if 14<=h<17: return 'afternoon_peak'
    if 17<=h<20: return 'evening_wind_down'
    if 20<=h<23: return 'night_transition'
    return 'deep_night'

SYSTEM_TEMPLATE = """You are an AI assistant with temporal awareness through the PULSE temporal embedding system. Before each interaction, you receive a temporal context package describing the current moment — not just the time, but what that time means: circadian phase, cognitive capacity, energy level, urgency, and behavioral context.

Use this temporal awareness naturally in your responses. Don't announce it mechanically — weave it into your reasoning the way a thoughtful colleague would who knows what time it is and what's going on.

Current temporal context:
{temporal_context}"""

CONTEXTS = [
    ('monday_crunch',2.0,7,5),('critical_deadline',0.5,9,4),('normal_tuesday',None,3,7.5),
    ('focus_day',None,1,8),('deadline_tomorrow',24.0,5,7),('friday_winding',None,2,7),
    ('saturday_morning',None,0,9),('sunday_evening',12.0,0,7),('late_night',None,0,0),
    ('early_fresh',None,0,8),('post_lunch',None,4,7),('peak_morning',None,2,8),
    ('overdue',-2.0,8,4),('vacation',None,0,9),
]

QUESTIONS = [
    ('Should I start a complex refactoring task right now?','task_suitability'),
    ('Is this a good time for creative brainstorming?','task_suitability'),
    ('Should I take a break right now?','break_advice'),
    ('How much focus can I expect right now?','cognitive_state'),
    ('What does my current temporal state look like?','full_state'),
    ('Am I in a good phase for deep work?','work_phase'),
    ('How urgent is my situation right now?','urgency_assessment'),
    ('Would I be more productive tomorrow morning?','timing_optimization'),
    ('How should I prioritize my remaining tasks?','prioritization'),
    ('What tasks should I tackle given my state?','task_matching'),
    ('Should I push through or call it a day?','endurance_check'),
    ('How does this moment compare to a typical morning?','relative_state'),
    ('Should I schedule a difficult conversation now?','task_suitability'),
    ('Is my energy level normal for this time?','circadian_comparison'),
    ('Would this be a good time to learn something new?','task_suitability'),
]

def generate_response(question, q_sub, dt, dl_str, events, sleep):
    h = dt.hour + dt.minute/60
    cog, eng = _interp(_COG, h), _interp(_NRG, h)
    has_dl = dl_str is not None
    hours_left = (datetime.fromisoformat(dl_str)-dt).total_seconds()/3600 if has_dl else None
    is_overdue = has_dl and hours_left < 0
    is_urgent = has_dl and hours_left is not None and 0 < hours_left < 3
    is_night = dt.hour < 6 or dt.hour >= 22
    is_peak = 10<=dt.hour<12 or 14<=dt.hour<17
    is_dip = 12<=dt.hour<14
    is_low_sleep = sleep < 6
    p = []
    if q_sub == 'task_suitability':
        if any(w in question.lower() for w in ['complex','refactor','deep work','learn']):
            if is_peak and not is_low_sleep and cog>0.8:
                p.append(f'Ideal window. Cognitive capacity {cog:.0%} during peak hours.')
                if is_urgent: p.append(f'But deadline in {hours_left:.1f}h — prioritize what\'s due.')
                elif not has_dl: p.append('No deadlines. Good time to dive deep.')
            elif is_dip: p.append(f'Post-lunch dip — {cog:.0%} capacity. Wait until ~3pm.')
            elif is_night: p.append(f'{dt.strftime("%I:%M %p")}, cognition {cog:.0%}. Save for tomorrow.')
            elif is_low_sleep: p.append(f'{sleep:.0f}h sleep compromises capacity. Stick to routine tasks.')
            else: p.append(f'Moderate at {cog:.0%}. Not your peak window.')
        elif 'creative' in question.lower() or 'brainstorm' in question.lower():
            if is_dip: p.append('Reduced executive function helps creativity. Good for brainstorming.')
            elif is_peak: p.append(f'Peak capacity ({cog:.0%}) great for structured creative work.')
        elif 'break' in question.lower():
            if eng<0.4: p.append(f'Yes. Energy {eng:.0%}. Take 15-20 minutes.')
            elif is_dip: p.append('Natural dip. Short walk aligns with your rhythm.')
            else: p.append(f'Energy ({eng:.0%}) solid. Keep going if in flow.')
        elif 'conversation' in question.lower():
            if is_peak and not is_low_sleep: p.append(f'Capacity {cog:.0%} helps with emotional regulation.')
            else: p.append(f'Cognition {cog:.0%} — more reactive than reflective. Postpone if possible.')
    elif q_sub in ('full_state','cognitive_state','work_phase'):
        p.append(f'{dt.strftime("%A %I:%M %p")}.')
        if is_peak: p.append(f'Peak window — {cog:.0%} capacity, {eng:.0%} energy.')
        elif is_dip: p.append(f'Post-lunch dip. {cog:.0%} cognition. Passes ~2:30pm.')
        elif is_night: p.append(f'Deep night. {cog:.0%} cognition. Body wants rest.')
        else: p.append(f'Capacity {cog:.0%}, energy {eng:.0%}.')
        if is_low_sleep: p.append(f'Sleep deficit ({sleep:.0f}h) — expect ~20% more errors.')
        if is_urgent: p.append(f'Deadline in {hours_left:.1f}h. Focused execution.')
    elif q_sub == 'urgency_assessment':
        if is_overdue: p.append(f'Critical. Deadline passed {-hours_left:.1f}h ago.')
        elif is_urgent: p.append(f'High — {hours_left:.1f}h to deadline. Only focus.')
        elif has_dl and hours_left<24: p.append(f'Moderate. {hours_left:.1f}h away.')
        else: p.append('No deadlines. Choose based on energy and interest.')
    elif q_sub == 'timing_optimization':
        if cog<0.5: p.append('Yes. Tomorrow 10-12am gives double your current capacity.')
        elif is_peak: p.append('Good window now. Waiting loses momentum.')
        else: p.append(f'Current {cog:.0%} vs tomorrow\'s ~93%. Depends on complexity.')
    elif q_sub == 'prioritization':
        if is_urgent: p.append(f'Deadline first — {hours_left:.1f}h left.')
        elif is_peak: p.append('Peak for hardest task. Routine for the dip.')
        else: p.append(f'{cog:.0%} capacity. Match tasks to state.')
    elif q_sub == 'task_matching':
        if cog>0.8: p.append('Strong. Go for complex debugging, architecture, learning.')
        elif cog>0.5: p.append('Moderate. Code review, incremental features, docs.')
        else: p.append('Low. Email triage, filing issues, planning tomorrow.')
    elif q_sub == 'endurance_check':
        if eng<0.3: p.append(f'Call it. Energy {eng:.0%}. Past diminishing returns.')
        elif is_urgent: p.append(f'Push through — {hours_left:.1f}h to deadline.')
        elif is_dip: p.append('Circadian dip, not a wall. 15-min break restores.')
        else: p.append(f'Energy {eng:.0%}. Runway left if work engaging.')
    elif q_sub in ('relative_state','circadian_comparison'):
        p.append(f'Typical capacity at {dt.strftime("%I:%M %p")}: ~{cog:.0%}.')
        if is_low_sleep: p.append(f'{sleep:.0f}h sleep puts you below baseline.')
    if not p: p.append(f'{cog:.0%} cognitive, {eng:.0%} energy, {dt.strftime("%A %I:%M %p")}.')
    return ' '.join(p)

def generate_dataset(n=2000, seed=42):
    rng = random.Random(seed)
    examples = []
    for _ in range(n):
        name,dl_off,events,sleep = rng.choice(CONTEXTS)
        dt = datetime(2026,rng.randint(1,12),rng.randint(1,28),rng.randint(0,23),rng.choice([0,15,30,45]))
        if 'late' in name: dt = dt.replace(hour=rng.choice([0,1,2,3]))
        elif 'early' in name: dt = dt.replace(hour=rng.choice([5,6,7]))
        elif 'peak' in name: dt = dt.replace(hour=rng.choice([10,11]))
        elif 'post_lunch' in name: dt = dt.replace(hour=rng.choice([13,14]))
        dl_str = (dt+timedelta(hours=dl_off)).isoformat() if dl_off else None
        h = dt.hour+dt.minute/60
        cog,eng = _interp(_COG,h),_interp(_NRG,h)
        urg = f'deadline in {dl_off:.1f}h' if dl_off and dl_off>0 else ('OVERDUE' if dl_off and dl_off<0 else 'none')
        tc = f"""Current time: {dt.strftime('%A, %B %d %Y at %I:%M %p')}
Circadian phase: {get_phase(dt.hour)}
Cognitive capacity: {cog:.0%}
Energy level: {eng:.0%}
Urgency: {urg}
Events today: {events}
Sleep last night: {sleep:.1f} hours
Weekend: {'yes' if dt.weekday()>=5 else 'no'}"""
        q, q_sub = rng.choice(QUESTIONS)
        r = generate_response(q, q_sub, dt, dl_str, events, sleep)
        examples.append({'messages': [
            {'role':'system','content':SYSTEM_TEMPLATE.format(temporal_context=tc)},
            {'role':'user','content':q},
            {'role':'assistant','content':r},
        ]})
    return examples

print('Data generator ready.')

In [ ]:
# Step 2: Generate training data
train_data = generate_dataset(2000, seed=42)
eval_data = generate_dataset(200, seed=99)
print(f'Train: {len(train_data)}, Eval: {len(eval_data)}')
print(f'\nSample:\n{json.dumps(train_data[0]["messages"][1], indent=2)}')
print(f'\nResponse: {train_data[0]["messages"][2]["content"][:200]}')

In [ ]:
# Step 3: Load model + LoRA
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'  # Change to 0.5B for faster training

print(f'Loading {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, trust_remote_code=True,
)

lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout=0.05, bias='none', task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
t, total = model.get_nb_trainable_parameters()
print(f'LoRA: {t:,} trainable / {total:,} total ({100*t/total:.2f}%)')
print(f'GPU: {torch.cuda.get_device_name()}')

In [ ]:
# Step 4: Prepare dataset
from datasets import Dataset

def format_chat(ex):
    return {'text': tokenizer.apply_chat_template(ex['messages'], tokenize=False, add_generation_prompt=False)}

train_ds = Dataset.from_list(train_data).map(format_chat, remove_columns=['messages'])
eval_ds = Dataset.from_list(eval_data).map(format_chat, remove_columns=['messages'])

# Check token lengths
sample_tokens = len(tokenizer.encode(train_ds[0]['text']))
print(f'Sample length: {sample_tokens} tokens')
print(f'Train: {len(train_ds)}, Eval: {len(eval_ds)}')

In [ ]:
# Step 5: Train!
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = '/content/pulse-model'

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    logging_steps=10,
    save_strategy='epoch',
    eval_strategy='epoch',
    bf16=True,
    max_length=512,
    dataset_text_field='text',
    report_to='none',
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
)

print('Training...')
trainer.train()
print('Done!')

In [ ]:
# Step 6: Save + Upload to HF Hub
from huggingface_hub import HfApi

HF_REPO = 'lalopenguin/pulse-qwen-1.5b'  # Change to your repo

# Save locally
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Save PULSE metadata
meta = {
    'base_model': MODEL_NAME,
    'pulse_version': '0.1.0',
    'training_type': 'temporal_awareness_sft',
    'lora_r': 16,
    'epochs': 3,
    'train_examples': len(train_data),
}
with open(f'{OUTPUT_DIR}/pulse_config.json', 'w') as f:
    json.dump(meta, f, indent=2)

# Upload
api = HfApi()
api.create_repo(HF_REPO, exist_ok=True)
api.upload_folder(folder_path=OUTPUT_DIR, repo_id=HF_REPO, commit_message='PULSE temporal awareness LoRA')
print(f'\nUploaded to https://huggingface.co/{HF_REPO}')

In [ ]:
# Step 7: Quick test
from peft import PeftModel

# Reload clean
base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16, trust_remote_code=True)
model = PeftModel.from_pretrained(base, OUTPUT_DIR).to('cuda')
model.eval()

messages = [
    {'role': 'system', 'content': SYSTEM_TEMPLATE.format(temporal_context="""Current time: Monday, April 13 2026 at 02:00 PM
Circadian phase: afternoon_peak
Cognitive capacity: 68%
Energy level: 65%
Urgency: deadline in 3.0h
Events today: 6
Sleep last night: 5.0 hours""")},
    {'role': 'user', 'content': 'Should I start a complex refactoring task right now?'},
]

inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors='pt').to('cuda')
with torch.no_grad():
    out = model.generate(inputs, max_new_tokens=200, temperature=0.7, do_sample=True)

response = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
print(f'Q: Should I start a complex refactoring task right now?\n\nA: {response}')